# DeepGuard AI — Notebook 3 · Streamlit UI + Auth

The runtime user interface, authentication layer, and Streamlit theme.

## To run
```bash
cd C:\dl\deepfake
py -3.11 -m streamlit run app.py --server.headless=true --server.port=8501
```
Then open http://localhost:8501 and log in with `admin` / `Admin@123`.

## `.streamlit/config.toml` (dark theme)

```toml
[theme]
base = "dark"
primaryColor = "#3b82f6"
backgroundColor = "#0b1220"
secondaryBackgroundColor = "#111a2e"
textColor = "#e2e8f0"
font = "sans serif"

[server]
maxUploadSize = 200
```

## What the UI shows
- Login page with hacker-grid background
- Sidebar navigation: **Analyze**, **Dashboard**, **History**, **Admin**
- Analyze: upload -> preview -> verdict tile (LIKELY AUTHENTIC / MANIPULATION SUSPECTED / INCONCLUSIVE / NO SUITABLE FACE)
- Per-face grid with individual Grad-CAM heatmaps for multi-person photos
- Video: per-person track results + per-frame timeline + top-5 suspicious frames
- Forensic panel: SHA-256, perceptual hash, model version
- Download evidence report (PDF)
- Admin: user management + system stats
- History: past analyses per user


## `auth.py` — PBKDF2-based auth + SQLite audit trail

In [ ]:
# auth.py — Authentication & Authorization for Deepfake Detection UI
# Storage: SQLite (stdlib) · Hashing: PBKDF2-HMAC-SHA256 + salt

import sqlite3
import hashlib
import secrets
from datetime import datetime

DB_PATH = "deepfake.db"


# ─────────────────────────────────────────────────────────────────────────────
# DB Initialisation
# ─────────────────────────────────────────────────────────────────────────────

def init_db() -> None:
    """Create tables and seed the default admin account."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            username      TEXT    UNIQUE NOT NULL,
            email         TEXT    UNIQUE NOT NULL,
            password_hash TEXT    NOT NULL,
            salt          TEXT    NOT NULL,
            role          TEXT    DEFAULT 'analyst',
            created_at    TEXT    NOT NULL,
            last_login    TEXT,
            is_active     INTEGER DEFAULT 1
        )
    """)

    c.execute("""
        CREATE TABLE IF NOT EXISTS analyses (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id     INTEGER NOT NULL,
            filename    TEXT    NOT NULL,
            file_type   TEXT    NOT NULL,
            verdict     TEXT    NOT NULL,
            score       REAL    NOT NULL,
            confidence  REAL    NOT NULL,
            sha256      TEXT    NOT NULL,
            model_name  TEXT    NOT NULL,
            elapsed_ms  REAL    NOT NULL,
            analyzed_at TEXT    NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    """)

    conn.commit()

    # Seed default admin
    c.execute("SELECT id FROM users WHERE username = 'admin'")
    if not c.fetchone():
        _create_user(conn, "admin", "admin@deepfake.ai", "Admin@123", "admin")

    conn.close()


# ─────────────────────────────────────────────────────────────────────────────
# Internal helpers
# ─────────────────────────────────────────────────────────────────────────────

def _hash_password(password: str, salt: str) -> str:
    key = hashlib.pbkdf2_hmac(
        "sha256", password.encode(), salt.encode(), 200_000
    )
    return key.hex()


def _create_user(conn: sqlite3.Connection, username: str, email: str,
                 password: str, role: str = "analyst") -> bool:
    salt = secrets.token_hex(32)
    ph = _hash_password(password, salt)
    try:
        conn.execute(
            "INSERT INTO users (username,email,password_hash,salt,role,created_at) "
            "VALUES (?,?,?,?,?,?)",
            (username, email, ph, salt, role, datetime.now().isoformat()),
        )
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False


# ─────────────────────────────────────────────────────────────────────────────
# Public auth API
# ─────────────────────────────────────────────────────────────────────────────

def authenticate(username: str, password: str) -> dict | None:
    """Verify credentials. Returns user dict or None."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        "SELECT id,username,email,password_hash,salt,role,is_active "
        "FROM users WHERE username=?",
        (username,),
    )
    row = c.fetchone()
    if not row:
        conn.close()
        return None
    uid, uname, email, ph, salt, role, active = row
    if not active or _hash_password(password, salt) != ph:
        conn.close()
        return None
    conn.execute(
        "UPDATE users SET last_login=? WHERE id=?",
        (datetime.now().isoformat(), uid),
    )
    conn.commit()
    conn.close()
    return {"id": uid, "username": uname, "email": email, "role": role}


def register_user(username: str, email: str, password: str) -> tuple[bool, str]:
    """Register a new analyst account."""
    username = username.strip()
    email = email.strip().lower()
    if len(username) < 3:
        return False, "Username must be at least 3 characters."
    if len(password) < 8:
        return False, "Password must be at least 8 characters."
    if not any(c.isupper() for c in password):
        return False, "Password must contain at least one uppercase letter."
    if not any(c.isdigit() for c in password):
        return False, "Password must contain at least one digit."
    if "@" not in email or "." not in email:
        return False, "Invalid email address."
    conn = sqlite3.connect(DB_PATH)
    ok = _create_user(conn, username, email, password, "analyst")
    conn.close()
    return (True, "Account created! You can now sign in.") if ok \
        else (False, "Username or email already taken.")


# ─────────────────────────────────────────────────────────────────────────────
# User management (admin)
# ─────────────────────────────────────────────────────────────────────────────

def get_all_users() -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        "SELECT id,username,email,role,created_at,last_login,is_active "
        "FROM users ORDER BY created_at DESC"
    ).fetchall()
    conn.close()
    keys = ["id", "username", "email", "role", "created_at", "last_login", "is_active"]
    return [dict(zip(keys, r)) for r in rows]


def toggle_user_status(user_id: int) -> None:
    conn = sqlite3.connect(DB_PATH)
    conn.execute("UPDATE users SET is_active=1-is_active WHERE id=?", (user_id,))
    conn.commit()
    conn.close()


def change_user_role(user_id: int, role: str) -> None:
    conn = sqlite3.connect(DB_PATH)
    conn.execute("UPDATE users SET role=? WHERE id=?", (role, user_id))
    conn.commit()
    conn.close()


def delete_user(user_id: int) -> None:
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM users WHERE id=? AND username!='admin'", (user_id,))
    conn.commit()
    conn.close()


# ─────────────────────────────────────────────────────────────────────────────
# Analysis history
# ─────────────────────────────────────────────────────────────────────────────

def save_analysis(user_id: int, filename: str, file_type: str,
                  result: dict, sha256: str) -> None:
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT INTO analyses "
        "(user_id,filename,file_type,verdict,score,confidence,sha256,model_name,elapsed_ms,analyzed_at) "
        "VALUES (?,?,?,?,?,?,?,?,?,?)",
        (user_id, filename, file_type, result["verdict"], result["score"],
         result["confidence"], sha256, result["model_name"],
         result["elapsed_ms"], datetime.now().isoformat()),
    )
    conn.commit()
    conn.close()


def get_user_analyses(user_id: int, limit: int = 30) -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        "SELECT filename,file_type,verdict,score,confidence,sha256,"
        "model_name,elapsed_ms,analyzed_at "
        "FROM analyses WHERE user_id=? ORDER BY analyzed_at DESC LIMIT ?",
        (user_id, limit),
    ).fetchall()
    conn.close()
    keys = ["filename","file_type","verdict","score","confidence",
            "sha256","model_name","elapsed_ms","analyzed_at"]
    return [dict(zip(keys, r)) for r in rows]


def get_all_analyses(limit: int = 100) -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        "SELECT u.username,a.filename,a.file_type,a.verdict,a.score,"
        "a.confidence,a.sha256,a.model_name,a.elapsed_ms,a.analyzed_at "
        "FROM analyses a JOIN users u ON a.user_id=u.id "
        "ORDER BY a.analyzed_at DESC LIMIT ?",
        (limit,),
    ).fetchall()
    conn.close()
    keys = ["username","filename","file_type","verdict","score",
            "confidence","sha256","model_name","elapsed_ms","analyzed_at"]
    return [dict(zip(keys, r)) for r in rows]


def get_stats() -> dict:
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    total_users    = c.execute("SELECT COUNT(*) FROM users WHERE is_active=1").fetchone()[0]
    total_analyses = c.execute("SELECT COUNT(*) FROM analyses").fetchone()[0]
    total_fakes    = c.execute("SELECT COUNT(*) FROM analyses WHERE verdict='FAKE'").fetchone()[0]
    total_real     = c.execute("SELECT COUNT(*) FROM analyses WHERE verdict='REAL'").fetchone()[0]
    conn.close()
    return {
        "total_users":    total_users,
        "total_analyses": total_analyses,
        "total_fakes":    total_fakes,
        "total_real":     total_real,
        "detection_rate": round(total_fakes / total_analyses * 100, 1)
                          if total_analyses else 0.0,
    }


## `app.py` — full Streamlit application

In [ ]:
import streamlit as st
st.set_page_config(page_title="DeepGuard AI", page_icon="🛡", layout="wide", initial_sidebar_state="expanded")

import hashlib, os, tempfile, time
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from auth import (init_db, authenticate, register_user, get_all_users,
                  toggle_user_status, change_user_role, delete_user,
                  save_analysis, get_user_analyses, get_all_analyses, get_stats)
from predict import predict
from report import generate_report
from forensics import compute_phash

init_db()

# ── Session defaults ──────────────────────────────────────────────────────────
for k, v in {"user": None, "page": "analyze", "last_result": None,
             "last_file": None, "last_sha": None, "auth_tab": "login"}.items():
    if k not in st.session_state:
        st.session_state[k] = v

# ── Helpers ───────────────────────────────────────────────────────────────────
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""): h.update(chunk)
    return h.hexdigest()

def human_size(n):
    for u in ("B","KB","MB","GB"):
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} TB"

def device_label():
    try:
        import torch
        return f"CUDA · {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU"
    except: return "CPU"

def verdict_info(result):
    verdict = result.get("verdict", "INCONCLUSIVE")
    if verdict == "NO_FACE": return "#60a5fa", "🔵", "NO SUITABLE FACE"
    if verdict == "INCONCLUSIVE": return "#f59e0b", "🟡", "INCONCLUSIVE"
    if verdict == "FAKE": return "#ef4444", "🔴", "MANIPULATION SUSPECTED"
    return "#10b981", "🟢", "LIKELY AUTHENTIC"

def bgr2rgb(arr): return arr[..., ::-1]

# ── Global CSS ─────────────────────────────────────────────────────────────────
GLOBAL_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap');
html,body,[class*="css"]{font-family:'Inter',sans-serif;}
#MainMenu,footer,header,.stDeployButton{visibility:hidden;}
.main .block-container{padding:1.2rem 2rem 2rem;max-width:1300px;}
::-webkit-scrollbar{width:5px;} ::-webkit-scrollbar-track{background:#050d1a;}
::-webkit-scrollbar-thumb{background:#1e3a5f;border-radius:3px;}
[data-testid="stSidebar"]{background:linear-gradient(180deg,#06111f 0%,#030a14 100%) !important;border-right:1px solid #0f2040 !important;}
[data-testid="stFileUploadDropzone"]{background:rgba(10,22,44,.5)!important;border:2px dashed rgba(59,130,246,.35)!important;border-radius:14px!important;min-height:140px!important;transition:border-color .3s;}
[data-testid="stFileUploadDropzone"]:hover{border-color:#3b82f6!important;}
.stTextInput input{background:#0a1628!important;border:1px solid #1e3a5f!important;border-radius:8px!important;color:#f1f5f9!important;font-size:14px!important;}
.stTextInput input:focus{border-color:#3b82f6!important;box-shadow:0 0 0 3px rgba(59,130,246,.15)!important;}
.stButton>button{border-radius:9px!important;font-weight:600!important;transition:all .2s!important;}
.stButton>button[kind="primary"]{background:linear-gradient(135deg,#3b82f6,#6366f1)!important;border:none!important;box-shadow:0 4px 18px rgba(59,130,246,.35)!important;}
.stButton>button[kind="primary"]:hover{transform:translateY(-2px)!important;box-shadow:0 8px 24px rgba(59,130,246,.5)!important;}
.stTabs [data-baseweb="tab-list"]{background:#0a1628;border-radius:10px;padding:4px;gap:4px;}
.stTabs [data-baseweb="tab"]{border-radius:7px;color:#64748b;font-weight:500;}
.stTabs [aria-selected="true"]{background:#1e3a5f!important;color:#e2e8f0!important;}
.stProgress>div>div{background:linear-gradient(90deg,#3b82f6,#6366f1);border-radius:999px;}
@keyframes fadeUp{from{opacity:0;transform:translateY(16px)}to{opacity:1;transform:translateY(0)}}
@keyframes pulseBorder{0%,100%{box-shadow:0 0 0 0 var(--vc,#3b82f6)}50%{box-shadow:0 0 0 6px transparent}}
.anim-up{animation:fadeUp .4s ease both;}
/* Sidebar nav buttons — visible, styled, full-width */
[data-testid="stSidebar"] .stButton button{
  background:#0a1628 !important;
  border:1px solid #1e3a5f !important;
  color:#e2e8f0 !important;
  text-align:left !important;
  padding:10px 14px !important;
  font-size:14px !important;
  font-weight:500 !important;
  border-radius:8px !important;
  margin:2px 0 !important;
  transition:all .18s !important;
}
[data-testid="stSidebar"] .stButton button:hover{
  background:linear-gradient(90deg,#1e3a5f,#0f2040) !important;
  border-left:3px solid #3b82f6 !important;
  color:#fff !important;
}
</style>"""

AUTH_CSS = """
<style>
[data-testid="stSidebar"]{display:none!important;}
.main .block-container{padding:0!important;max-width:100%!important;}
.stApp{background:#020810!important;}
/* Animated hacker-style grid background */
.hacker-grid{position:fixed;inset:0;display:flex;flex-wrap:wrap;gap:2px;z-index:0;
  overflow:hidden;pointer-events:none;}
.hacker-grid::before{content:'';position:absolute;width:100%;height:100%;
  background:linear-gradient(#020810,#0044ff18,#020810);animation:gridSweep 5s linear infinite;z-index:1;}
@keyframes gridSweep{0%{transform:translateY(-100%)}100%{transform:translateY(100%)}}
.hacker-grid span{display:block;width:calc(6.25vw - 2px);height:calc(6.25vw - 2px);
  background:#080e1a;transition:1.5s;position:relative;z-index:2;}
/* Glassmorphism login card */
.glass-card{position:relative;z-index:100;background:rgba(8,18,38,0.75)!important;
  border:1px solid rgba(59,130,246,0.2)!important;border-radius:20px!important;
  backdrop-filter:blur(16px)!important;-webkit-backdrop-filter:blur(16px)!important;
  padding:8px 20px!important;box-shadow:0 8px 32px rgba(0,0,0,0.6),0 0 60px rgba(59,130,246,0.08)!important;}
@keyframes glowPulse{0%,100%{box-shadow:0 8px 32px rgba(0,0,0,0.6),0 0 60px rgba(59,130,246,0.08)}
  50%{box-shadow:0 8px 32px rgba(0,0,0,0.6),0 0 80px rgba(59,130,246,0.15)}}
.glass-card{animation:glowPulse 4s ease infinite;}
</style>"""

# Generate HTML spans for hacker grid background (16x12 = 192 tiles)
GRID_SPANS = '<div class="hacker-grid">' + '<span></span>' * 192 + '</div>'

# ── AUTH PAGE ─────────────────────────────────────────────────────────────────
def show_auth():
    st.markdown(GLOBAL_CSS + AUTH_CSS, unsafe_allow_html=True)
    # Animated hacker grid background
    st.markdown(GRID_SPANS, unsafe_allow_html=True)

    _, col, _ = st.columns([1, 1.2, 1])
    with col:
        st.markdown('<div class="glass-card">', unsafe_allow_html=True)

        st.markdown("""
        <div style="text-align:center;padding:32px 0 16px;">
          <div style="font-size:56px;filter:drop-shadow(0 0 24px #3b82f6cc);
            animation:float 3s ease-in-out infinite;">🛡</div>
          <div style="font-size:28px;font-weight:800;background:linear-gradient(135deg,#e2e8f0 30%,#3b82f6);
            -webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-top:10px;
            letter-spacing:.5px;">DeepGuard AI</div>
          <div style="font-size:13px;color:#475569;margin-top:4px;letter-spacing:.5px;">
            Forensic Deepfake Detection Platform</div>
        </div>
        <style>@keyframes float{0%,100%{transform:translateY(0)}50%{transform:translateY(-8px)}}</style>
        """, unsafe_allow_html=True)

        tab_login, tab_reg = st.tabs(["🔐 Sign In", "✨ Create Account"])

        with tab_login:
            st.markdown("<div style='height:12px'></div>", unsafe_allow_html=True)
            uname = st.text_input("Username", key="li_user", placeholder="Enter username")
            pwd   = st.text_input("Password", type="password", key="li_pwd", placeholder="Enter password")
            st.markdown("<div style='height:4px'></div>", unsafe_allow_html=True)
            if st.button("Sign In →", type="primary", use_container_width=True, key="btn_login"):
                if not uname or not pwd:
                    st.error("Please fill in all fields.")
                else:
                    user = authenticate(uname.strip(), pwd)
                    if user:
                        st.session_state.user = user
                        st.session_state.page = "dashboard"
                        st.rerun()
                    else:
                        st.error("Invalid credentials or account disabled.")
            st.markdown("""<div style="text-align:center;margin-top:16px;font-size:12px;color:#334155;">
              Default admin: <code style="color:#3b82f6">admin / Admin@123</code></div>""", unsafe_allow_html=True)

        with tab_reg:
            st.markdown("<div style='height:12px'></div>", unsafe_allow_html=True)
            r_user  = st.text_input("Username", key="rg_user", placeholder="Choose a username")
            r_email = st.text_input("Email",    key="rg_email", placeholder="your@email.com")
            r_pwd   = st.text_input("Password", type="password", key="rg_pwd",
                                    placeholder="Min 8 chars · uppercase · digit")
            r_pwd2  = st.text_input("Confirm Password", type="password", key="rg_pwd2",
                                    placeholder="Repeat password")
            st.markdown("<div style='height:4px'></div>", unsafe_allow_html=True)
            if st.button("Create Account →", type="primary", use_container_width=True, key="btn_reg"):
                if not all([r_user, r_email, r_pwd, r_pwd2]):
                    st.error("Please fill in all fields.")
                elif r_pwd != r_pwd2:
                    st.error("Passwords do not match.")
                else:
                    ok, msg = register_user(r_user, r_email, r_pwd)
                    (st.success if ok else st.error)(msg)

        st.markdown('</div>', unsafe_allow_html=True)  # close glass-card
        st.markdown("<div style='height:32px'></div>", unsafe_allow_html=True)

# ── SIDEBAR NAV ───────────────────────────────────────────────────────────────
def show_sidebar():
    u = st.session_state.user
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:20px 8px 4px;text-align:center;">
          <div style="font-size:32px;filter:drop-shadow(0 0 12px #3b82f688);">🛡</div>
          <div style="font-size:15px;font-weight:700;color:#3b82f6;letter-spacing:1px;margin-top:4px;">DeepGuard AI</div>
        </div>""", unsafe_allow_html=True)
        st.divider()

        nav_items = [("🔬", "Analyze",   "analyze"),
                     ("🏠", "Dashboard", "dashboard"),
                     ("📋", "History",   "history")]
        if u["role"] == "admin":
            nav_items.append(("👥", "Admin",    "admin"))

        # Real, visible, clickable navigation buttons
        for icon, label, key in nav_items:
            active = st.session_state.page == key
            btn_label = f"{'▸ ' if active else '  '}{icon}  {label}"
            if st.button(btn_label, key=f"nav_{key}", use_container_width=True):
                st.session_state.page = key
                st.rerun()

        st.divider()
        # User card
        role_color = "#f59e0b" if u["role"] == "admin" else "#3b82f6"
        st.markdown(f"""
        <div style="background:#0a1628;border:1px solid #1e3a5f;border-radius:10px;padding:12px 14px;margin:4px 0;">
          <div style="font-size:13px;font-weight:600;color:#e2e8f0;">👤 {u['username']}</div>
          <div style="font-size:11px;color:{role_color};margin-top:2px;text-transform:uppercase;
            letter-spacing:.8px;">{u['role']}</div>
          <div style="font-size:11px;color:#334155;margin-top:2px;">{u['email']}</div>
        </div>""", unsafe_allow_html=True)

        if st.button("🚪 Sign Out", use_container_width=True, key="btn_logout"):
            st.session_state.user = None
            st.session_state.last_result = None
            st.rerun()

# ── DASHBOARD ─────────────────────────────────────────────────────────────────
def show_dashboard():
    st.markdown("<h2 style='margin-bottom:4px;'>🏠 Dashboard</h2>", unsafe_allow_html=True)
    st.markdown("<p style='color:#64748b;margin-bottom:20px;'>System overview and recent activity</p>", unsafe_allow_html=True)

    stats = get_stats()
    c1,c2,c3,c4 = st.columns(4)
    def stat_card(col, icon, label, value, color):
        col.markdown(f"""
        <div style="background:linear-gradient(135deg,{color}18,{color}05);border:1px solid {color}40;
          border-radius:14px;padding:20px;text-align:center;height:110px;">
          <div style="font-size:28px;">{icon}</div>
          <div style="font-size:26px;font-weight:800;color:{color};margin-top:2px;">{value}</div>
          <div style="font-size:11px;color:#64748b;text-transform:uppercase;letter-spacing:.8px;">{label}</div>
        </div>""", unsafe_allow_html=True)

    stat_card(c1,"🔬","Total Analyses", stats["total_analyses"],"#3b82f6")
    stat_card(c2,"🔴","Fakes Detected",  stats["total_fakes"],   "#ef4444")
    stat_card(c3,"🟢","Verified Real",   stats["total_real"],    "#10b981")
    stat_card(c4,"📊","Detection Rate",  f"{stats['detection_rate']:.1f}%","#f59e0b")

    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("#### 📋 Recent Analyses")
    uid = st.session_state.user["id"]
    rows = get_user_analyses(uid, limit=10)
    if not rows:
        st.info("No analyses yet. Go to **Analyze** to get started.")
        return

    for r in rows:
        vc = "#ef4444" if r["verdict"]=="FAKE" else "#10b981"
        icon = "🔴" if r["verdict"]=="FAKE" else "🟢"
        st.markdown(f"""
        <div style="background:#0a1628;border:1px solid #1e3a5f;border-radius:10px;
          padding:12px 16px;margin:4px 0;display:flex;justify-content:space-between;align-items:center;">
          <div>
            <span style="font-size:13px;font-weight:600;color:#e2e8f0;">{r['filename']}</span>
            <span style="font-size:11px;color:#475569;margin-left:8px;">{r['file_type'].upper()}</span>
          </div>
          <div style="display:flex;align-items:center;gap:16px;">
            <span style="color:{vc};font-weight:700;font-size:13px;">{icon} {r['verdict']}</span>
            <span style="color:#64748b;font-size:12px;">{r['score']*100:.0f}% fake</span>
            <span style="color:#334155;font-size:11px;">{r['analyzed_at'][:16].replace('T',' ')}</span>
          </div>
        </div>""", unsafe_allow_html=True)

# ── VERDICT TILE ──────────────────────────────────────────────────────────────
def render_verdict(result):
    color, icon, label = verdict_info(result)
    score = result["score"] * 100
    conf  = result["confidence"] * 100
    st.markdown(f"""
    <div class="anim-up" style="--vc:{color};border:2px solid {color};border-radius:14px;
      padding:28px 32px;margin:16px 0;
      background:linear-gradient(135deg,{color}12,{color}04);
      box-shadow:0 0 30px {color}22;animation:pulseBorder 2.5s ease infinite;">
      <div style="font-size:11px;color:{color};letter-spacing:3px;font-weight:600;">▸ ANALYSIS VERDICT</div>
      <div style="font-size:38px;font-weight:800;color:{color};margin:6px 0 0;">{icon} {label}</div>
      <div style="display:flex;gap:32px;margin-top:14px;">
        <div><div style="font-size:22px;font-weight:800;color:{color}">{score:.1f}%</div>
          <div style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.6px;">Fake Probability</div></div>
        <div style="width:1px;background:#1e3a5f;"></div>
        <div><div style="font-size:22px;font-weight:800;color:{color}">{conf:.1f}%</div>
          <div style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.6px;">Confidence</div></div>
        <div style="width:1px;background:#1e3a5f;"></div>
        <div><div style="font-size:22px;font-weight:800;color:#94a3b8">{result['elapsed_ms']:.0f} ms</div>
          <div style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.6px;">Inference</div></div>
      </div>
    </div>""", unsafe_allow_html=True)
    st.progress(float(result["confidence"]))

# ── TIMELINE CHART ────────────────────────────────────────────────────────────
def render_timeline(result):
    st.markdown("#### 📈 Per-Frame Analysis Timeline")
    xs = np.array(result["frame_indices"])
    ys = np.array(result["per_frame"])
    fig, ax = plt.subplots(figsize=(10, 3))
    fig.patch.set_facecolor("#060f20")
    ax.set_facecolor("#0a1628")
    ax.fill_between(xs, ys, .5, where=(ys>.5), interpolate=True, color="#ef4444", alpha=.3)
    ax.fill_between(xs, ys, .5, where=(ys<=.5), interpolate=True, color="#10b981", alpha=.3)
    ax.plot(xs, ys, color="#60a5fa", lw=1.8, zorder=3)
    ax.scatter(xs, ys, s=14, color="#3b82f6", zorder=4, linewidths=0)
    ax.axhline(.35, color="#f59e0b", lw=1.2, ls="--", alpha=.8)
    ax.text(xs[-1], .38, "  threshold", color="#f59e0b", fontsize=8, ha="right")
    if result.get("top_frames"):
        for tf in result["top_frames"]:
            ax.axvline(tf["frame_index"], color="#ef4444", lw=.8, alpha=.45, ls=":")
    ax.set_xlim(xs[0], xs[-1]); ax.set_ylim(-.05, 1.08)
    ax.set_xlabel("Frame index", color="#475569", fontsize=8)
    ax.set_ylabel("Fake prob.", color="#475569", fontsize=8)
    ax.tick_params(colors="#334155", labelsize=7)
    for sp in ax.spines.values(): sp.set_color("#1e3a5f")
    fp = mpatches.Patch(color="#ef4444", alpha=.6, label="Suspicious")
    rp = mpatches.Patch(color="#10b981", alpha=.6, label="Clean")
    ax.legend(handles=[fp, rp], framealpha=.15, labelcolor="#e2e8f0",
              fontsize=8, facecolor="#0a1628", edgecolor="#1e3a5f", loc="upper right")
    ax.set_title(f"Sampled @ {result['fps_sampled']:.1f} fps · {len(xs)} frames · median {result['score']*100:.1f}%",
                 color="#64748b", fontsize=8, pad=6)
    fig.tight_layout(pad=1)
    st.pyplot(fig, use_container_width=True)
    plt.close(fig)

def render_track_summary(result):
    tracks = result.get("tracks") or []
    if not tracks:
        st.warning("No sufficiently clear face track was available. Video verdict is inconclusive; integrity checks remain available.")
        return
    st.markdown("#### Per-Person Track Summary")
    cols = st.columns(min(4, len(tracks)))
    for i, track in enumerate(tracks):
        verdict = track.get("verdict", "INCONCLUSIVE")
        color = "#ef4444" if verdict == "FAKE" else ("#10b981" if verdict == "REAL" else "#f59e0b")
        with cols[i % len(cols)]:
            st.markdown(f"""<div style="border:1px solid {color};border-radius:9px;padding:10px;text-align:center;">
              <div style="font-size:11px;color:#64748b;">FACE TRACK {track['face_id']}</div>
              <div style="font-weight:800;color:{color};">{verdict}</div>
              <div style="font-size:12px;color:#94a3b8;">{track['score']*100:.1f}% · {track['observations']} frames</div>
            </div>""", unsafe_allow_html=True)
# ── TOP FRAMES (VIDEO) ────────────────────────────────────────────────────────
def render_top_frames(result):
    st.markdown("#### 🔍 Most Suspicious Frames — Grad-CAM Overlays")
    frames = result.get("top_frames", [])
    if not frames: return
    cols = st.columns(len(frames))
    for col, tf in zip(cols, frames):
        with col:
            img = tf.get("heatmap_bgr")
            if img is None:
                img = tf.get("face_bgr")
            if img is not None:
                st.image(bgr2rgb(img), use_container_width=True)
            vc = "#ef4444" if tf["score"]>=.35 else "#10b981"
            st.markdown(f"""<div style="text-align:center;margin-top:4px;">
              <div style="font-size:19px;font-weight:800;color:{vc}">{tf['score']*100:.0f}%</div>
              <div style="font-size:10px;color:#475569">Frame {tf['frame_index']}</div>
            </div>""", unsafe_allow_html=True)

# ── IMAGE PANELS ──────────────────────────────────────────────────────────────
def render_image_panels(result):
    if result.get("verdict") == "NO_FACE":
        st.warning("No suitable face was detected for facial deepfake analysis. File-integrity checks and hashes are still available below.")
        return
    st.markdown("#### 🧠 Primary Face + Grad-CAM Attention")
    c1, c2 = st.columns(2)
    with c1:
        st.markdown('<p style="text-align:center;font-size:11px;color:#475569;letter-spacing:1px;text-transform:uppercase;">Primary Face Crop (Largest)</p>', unsafe_allow_html=True)
        face = result.get("face_crop_bgr")
        if face is not None: st.image(bgr2rgb(face), use_container_width=True, caption="224x224 extracted face")
        else: st.warning("No face detected.")
    with c2:
        st.markdown('<p style="text-align:center;font-size:11px;color:#475569;letter-spacing:1px;text-transform:uppercase;">Grad-CAM Heatmap</p>', unsafe_allow_html=True)
        hmap = result.get("heatmap_bgr")
        if hmap is not None: st.image(bgr2rgb(hmap), use_container_width=True, caption="Model attention overlay")
        else: st.info("Heatmap unavailable.")

    # ═══ ALL detected faces with per-face heatmaps ═══
    all_faces = result.get("all_faces") or []
    if len(all_faces) >= 2:
        st.markdown("#### 🔍 All Detected Faces — Per-Face Grad-CAM")
        st.caption(f"{len(all_faces)} faces detected · each face scored + heatmapped independently")
        cols_per_row = 4
        for row_start in range(0, len(all_faces), cols_per_row):
            row = all_faces[row_start:row_start + cols_per_row]
            cols = st.columns(len(row))
            for col, f in zip(cols, row):
                with col:
                    fs = float(f.get("score", 0))
                    bl = float(f.get("blur", 0))
                    is_pri = f.get("is_primary", False)
                    reliable = bl >= 50
                    color = "#f59e0b" if not reliable else ("#ef4444" if fs >= 0.35 else "#10b981")
                    star  = " ★ PRIMARY" if is_pri else ""
                    bl_lbl = "INCONCLUSIVE - blurry" if bl < 50 else "sufficient quality"
                    # side-by-side: crop + heatmap
                    sub1, sub2 = st.columns(2)
                    crop = f.get("crop_bgr")
                    heat = f.get("heatmap_bgr")
                    if crop is not None: sub1.image(bgr2rgb(crop), use_container_width=True)
                    if heat is not None: sub2.image(bgr2rgb(heat), use_container_width=True)
                    st.markdown(f"""<div style="text-align:center;font-family:monospace;
                        font-size:12px;line-height:1.4;margin-top:4px;">
                        <span style="color:{color};font-weight:700;">{fs*100:.0f}%</span>
                        <span style="color:#64748b;"> fake · {bl_lbl}</span>
                        <span style="color:#3b82f6;">{star}</span>
                    </div>""", unsafe_allow_html=True)

# ── ANALYZE PAGE ──────────────────────────────────────────────────────────────
def show_analyze():
    st.markdown("<h2 style='margin-bottom:4px;'>🔬 Analyze Evidence</h2>", unsafe_allow_html=True)
    st.markdown("<p style='color:#64748b;margin-bottom:20px;'>Upload an image or video for deepfake detection analysis.</p>", unsafe_allow_html=True)

    # ── Prominent upload section with visible label + button ──────────────
    st.markdown("""
    <div style="background:linear-gradient(135deg,#0a1628 0%,#06111f 100%);
      border:1px solid #1e3a5f;border-radius:14px;padding:18px 22px;margin-bottom:12px;">
      <div style="display:flex;align-items:center;gap:14px;">
        <div style="font-size:32px;">⬆</div>
        <div style="flex:1;">
          <div style="font-size:16px;font-weight:700;color:#e2e8f0;">
            Choose an image or video to analyze
          </div>
          <div style="font-size:12px;color:#64748b;margin-top:3px;">
            Click <b style="color:#3b82f6;">"Browse files"</b> below or drag &amp; drop into the box.
            Supported: JPG · PNG · WEBP · MP4 · MOV · AVI · MKV (max 200 MB)
          </div>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    uploaded = st.file_uploader(
        "📁 Upload evidence file (image or video)",
        type=["jpg","jpeg","png","webp","mp4","mov","avi","mkv"],
        help="Click 'Browse files' or drag & drop. Max 200 MB.",
        # NOTE: label_visibility left as default ("visible") so the button/label is obvious.
    )

    if not uploaded:
        st.info("👆  Waiting for a file — use the **Browse files** button or drag one into the box above.")
        return

    # Save to temp
    ext = os.path.splitext(uploaded.name)[1].lower()
    is_video = ext in {".mp4",".mov",".avi",".mkv"}
    with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
        tmp.write(uploaded.getvalue())
        tmp_path = tmp.name

    # Hashes
    sha  = sha256_file(tmp_path)
    try:    phash = compute_phash(tmp_path)
    except: phash = "unavailable"

    # Preview
    st.markdown("#### 🎬 Evidence Preview" if is_video else "#### 🖼 Evidence Preview")
    with st.container():
        if is_video: st.video(tmp_path)
        else:        st.image(tmp_path, use_container_width=True, caption=uploaded.name)

    # Run model
    with st.spinner("🔬 Analyzing evidence…  This may take a moment."):
        result = predict(tmp_path)

    # Save to DB
    save_analysis(st.session_state.user["id"], uploaded.name,
                  "video" if is_video else "image", result, sha)

    st.session_state.last_result = result
    st.session_state.last_sha    = sha

    # Verdict
    render_verdict(result)

    # Kind-specific panels
    if result["kind"] == "video":
        render_timeline(result)
        render_track_summary(result)
        st.markdown("<br>", unsafe_allow_html=True)
        render_top_frames(result)
    else:
        render_image_panels(result)

    # Forensic details
    st.divider()
    st.markdown("#### 🔒 Forensic Chain-of-Custody")
    fc1, fc2 = st.columns(2)
    with fc1:
        st.markdown('<div style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;margin-bottom:4px;">SHA-256 Hash</div>', unsafe_allow_html=True)
        st.code(sha, language=None)
    with fc2:
        st.markdown('<div style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;margin-bottom:4px;">Perceptual Hash</div>', unsafe_allow_html=True)
        st.code(phash, language=None)

    mc1, mc2, mc3 = st.columns(3)
    mc1.metric("Model", result["model_name"])
    mc2.metric("Version", result["model_version"])
    mc3.metric("Device", device_label())

    # Download report
    st.divider()
    col_dl, col_info = st.columns([2, 3])
    with col_dl:
        if st.button("📄 Generate Evidence Report (PDF)", type="primary",
                     use_container_width=True, key="btn_report"):
            with st.spinner("Generating PDF…"):
                pdf = generate_report(input_path=tmp_path, verdict=result,
                                      sha256=sha, phash=phash)
            st.download_button("💾 Save PDF", data=pdf,
                               file_name=f"{uploaded.name}.report.pdf",
                               mime="application/pdf",
                               use_container_width=True, key="btn_dl")
    with col_info:
        color, _, label = verdict_info(result)
        st.markdown(f"""<div style="font-size:12px;color:#475569;line-height:2;padding-top:6px;">
          📋 Verdict: <span style="color:{color};font-weight:700">{label}</span><br>
          📁 File: <span style="color:#e2e8f0">{uploaded.name}</span> ({human_size(uploaded.size)})<br>
          ⚡ Inference: <span style="color:#e2e8f0">{result['elapsed_ms']:.0f} ms</span>
        </div>""", unsafe_allow_html=True)

# ── HISTORY PAGE ──────────────────────────────────────────────────────────────
def show_history():
    st.markdown("<h2 style='margin-bottom:4px;'>📋 Analysis History</h2>", unsafe_allow_html=True)
    st.markdown("<p style='color:#64748b;margin-bottom:20px;'>Your past deepfake detection analyses.</p>", unsafe_allow_html=True)

    uid = st.session_state.user["id"]
    rows = get_user_analyses(uid, limit=50)
    if not rows:
        st.info("No analyses yet. Head to **Analyze** to get started.")
        return

    filter_col, _ = st.columns([1, 3])
    with filter_col:
        f = st.selectbox("Filter by verdict", ["All", "FAKE", "REAL"], key="hist_filter")

    filtered = rows if f == "All" else [r for r in rows if r["verdict"] == f]

    # Header
    st.markdown("""
    <div style="display:grid;grid-template-columns:2fr 1fr 1fr 1fr 1fr 1fr;
      gap:8px;padding:8px 16px;background:#0a1628;border-radius:8px 8px 0 0;
      border-bottom:1px solid #1e3a5f;margin-top:12px;">
      <span style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;">Filename</span>
      <span style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;">Type</span>
      <span style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;">Verdict</span>
      <span style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;">Score</span>
      <span style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;">Confidence</span>
      <span style="font-size:11px;color:#475569;text-transform:uppercase;letter-spacing:.8px;">Date</span>
    </div>""", unsafe_allow_html=True)

    for i, r in enumerate(filtered):
        vc = "#ef4444" if r["verdict"]=="FAKE" else "#10b981"
        bg = "#06111f" if i%2==0 else "#080f1a"
        st.markdown(f"""
        <div style="display:grid;grid-template-columns:2fr 1fr 1fr 1fr 1fr 1fr;
          gap:8px;padding:10px 16px;background:{bg};border-bottom:1px solid #0f2040;align-items:center;">
          <span style="font-size:13px;color:#e2e8f0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;"
                title="{r['filename']}">{r['filename']}</span>
          <span style="font-size:11px;color:#475569;text-transform:uppercase;">{r['file_type']}</span>
          <span style="font-size:12px;font-weight:700;color:{vc};">{r['verdict']}</span>
          <span style="font-size:13px;color:#e2e8f0;">{r['score']*100:.0f}%</span>
          <span style="font-size:13px;color:#e2e8f0;">{r['confidence']*100:.0f}%</span>
          <span style="font-size:11px;color:#334155;">{r['analyzed_at'][:16].replace('T',' ')}</span>
        </div>""", unsafe_allow_html=True)

# ── ADMIN PAGE ────────────────────────────────────────────────────────────────
def show_admin():
    if st.session_state.user["role"] != "admin":
        st.error("⛔ Access denied. Admin role required.")
        return

    st.markdown("<h2 style='margin-bottom:4px;'>👥 Admin Panel</h2>", unsafe_allow_html=True)
    st.markdown("<p style='color:#64748b;margin-bottom:20px;'>Manage users and monitor system activity.</p>", unsafe_allow_html=True)

    # Stats
    stats = get_stats()
    c1,c2,c3,c4 = st.columns(4)
    def s(col,icon,label,val,color):
        col.markdown(f"""<div style="background:{color}12;border:1px solid {color}30;border-radius:10px;
          padding:16px;text-align:center;"><div style="font-size:22px;">{icon}</div>
          <div style="font-size:22px;font-weight:800;color:{color}">{val}</div>
          <div style="font-size:10px;color:#475569;text-transform:uppercase;letter-spacing:.7px">{label}</div>
        </div>""", unsafe_allow_html=True)
    s(c1,"👥","Active Users",  stats["total_users"],    "#3b82f6")
    s(c2,"🔬","Total Analyses",stats["total_analyses"], "#6366f1")
    s(c3,"🔴","Fakes Found",   stats["total_fakes"],    "#ef4444")
    s(c4,"📊","Detection Rate",f"{stats['detection_rate']:.1f}%","#f59e0b")

    st.markdown("<br>#### 👤 User Management", unsafe_allow_html=True)

    users = get_all_users()
    for u in users:
        is_admin_acc = u["username"] == "admin"
        sc = "#10b981" if u["is_active"] else "#ef4444"
        uc = "#f59e0b" if u["role"]=="admin" else "#3b82f6"
        with st.container():
            col_info, col_role, col_act = st.columns([4, 2, 2])
            with col_info:
                st.markdown(f"""
                <div style="background:#0a1628;border:1px solid #1e3a5f;border-radius:10px;
                  padding:12px 16px;">
                  <span style="font-weight:700;color:#e2e8f0">{u['username']}</span>
                  <span style="font-size:11px;color:{uc};margin-left:8px;text-transform:uppercase;
                    letter-spacing:.8px;background:{uc}18;padding:2px 8px;border-radius:4px;">{u['role']}</span>
                  <span style="font-size:11px;color:{sc};margin-left:8px;">
                    {'● Active' if u['is_active'] else '● Inactive'}</span><br>
                  <span style="font-size:12px;color:#475569">{u['email']}</span>
                  <span style="font-size:11px;color:#334155;margin-left:12px;">
                    Last login: {u['last_login'][:16].replace('T',' ') if u['last_login'] else 'Never'}</span>
                </div>""", unsafe_allow_html=True)
            with col_role:
                if not is_admin_acc:
                    new_role = st.selectbox("Role", ["analyst","admin"],
                                            index=0 if u["role"]=="analyst" else 1,
                                            key=f"role_{u['id']}")
                    if new_role != u["role"]:
                        change_user_role(u["id"], new_role); st.rerun()
            with col_act:
                if not is_admin_acc:
                    label = "🔒 Disable" if u["is_active"] else "✅ Enable"
                    if st.button(label, key=f"tog_{u['id']}", use_container_width=True):
                        toggle_user_status(u["id"]); st.rerun()

    st.markdown("<br>#### 📋 All Analyses", unsafe_allow_html=True)
    all_rows = get_all_analyses(limit=30)
    for i, r in enumerate(all_rows):
        vc = "#ef4444" if r["verdict"]=="FAKE" else "#10b981"
        bg = "#06111f" if i%2==0 else "#080f1a"
        st.markdown(f"""
        <div style="display:grid;grid-template-columns:1fr 2fr 1fr 1fr 1fr;
          gap:8px;padding:9px 16px;background:{bg};border-bottom:1px solid #0f2040;">
          <span style="font-size:12px;color:#3b82f6">{r['username']}</span>
          <span style="font-size:12px;color:#e2e8f0;overflow:hidden;text-overflow:ellipsis;"
                title="{r['filename']}">{r['filename']}</span>
          <span style="font-size:12px;font-weight:700;color:{vc}">{r['verdict']}</span>
          <span style="font-size:12px;color:#94a3b8">{r['score']*100:.0f}%</span>
          <span style="font-size:11px;color:#334155">{r['analyzed_at'][:16].replace('T',' ')}</span>
        </div>""", unsafe_allow_html=True)

# ── MAIN ENTRY ────────────────────────────────────────────────────────────────
def main():
    st.markdown(GLOBAL_CSS, unsafe_allow_html=True)

    if not st.session_state.user:
        show_auth()
        return

    show_sidebar()

    page = st.session_state.page
    if page == "dashboard": show_dashboard()
    elif page == "analyze":  show_analyze()
    elif page == "history":  show_history()
    elif page == "admin":    show_admin()

if __name__ == "__main__":
    main()
